# 10 — CNN sobre mel-spectrograma com **CV repetido**

**Atualização metodológica:** versão anterior usava apenas holdout 80/20 (21 pacientes), criando comparação injusta com modelos clássicos avaliados em CV repetido. Esta versão aplica **3 repetições × 3-fold paciente-estratificado** = 9 folds independentes.

**Setup mantido:**
- Input: log-mel-spec 64 mels × 128 frames
- CNN pequena: 3 conv blocks + GAP + dense (~72k params)
- Loss: CrossEntropy com `class_weight` balanced
- 20 epochs por fold, lr=1e-3, AdamW, cosine LR schedule
- **9 fits totais** (vs 1 do holdout original) → tempo ~10-15 min em CPU

In [1]:
import sys, time, warnings
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from src import config, metadata
from src.plotting import set_style, savefig
from src.bootstrap import bootstrap_ci, fmt_ci

warnings.filterwarnings('ignore')
set_style()
torch.manual_seed(42); np.random.seed(42)
print(f'torch {torch.__version__} | device: cpu')

torch 2.12.0+cpu | device: cpu


In [2]:
rec_df = metadata.build_recordings_dataframe()
rec_df = metadata.filter_target_classes(rec_df).reset_index(drop=True)
le = LabelEncoder()
rec_df['label_int'] = le.fit_transform(rec_df['diagnosis'])
CLASSES = list(le.classes_)
HEALTHY_IDX = CLASSES.index('Healthy')
print(f'{len(rec_df)} gravações, {rec_df["patient_id"].nunique()} pacientes')
print(f'Classes: {CLASSES}')

851 gravações, 104 pacientes
Classes: ['COPD', 'Healthy', 'URTI']


## 1. Carregar mel-spectrogramas (cache do nb10 original)

In [3]:
MEL_PATH = config.PROCESSED_DIR / 'mel_specs.npz'
N_MELS, N_FRAMES, SR, HOP = 64, 128, 22050, 512

def make_logmel(y, sr=SR):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048,
                                        hop_length=HOP, n_mels=N_MELS, fmax=sr//2)
    return librosa.power_to_db(S, ref=np.max).astype(np.float32)

def fit_frames(spec, n_frames=N_FRAMES):
    if spec.shape[1] >= n_frames:
        start = (spec.shape[1] - n_frames) // 2
        return spec[:, start:start+n_frames]
    reps = int(np.ceil(n_frames / spec.shape[1]))
    return np.tile(spec, reps)[:, :n_frames]

if MEL_PATH.exists():
    arr = np.load(MEL_PATH)
    specs, filenames = arr['specs'], arr['filenames']
    # Mapear specs para a ordem do rec_df atual
    fname_to_idx = {f: i for i, f in enumerate(filenames)}
    specs_ordered = np.array([specs[fname_to_idx[f]] for f in rec_df['filename']])
    specs = specs_ordered
    print(f'Cache carregado: {specs.shape}')
else:
    print('Gerando mel-specs (sem cache)...')
    specs = np.zeros((len(rec_df), N_MELS, N_FRAMES), dtype=np.float32)
    from tqdm.auto import tqdm
    for i, row in tqdm(rec_df.iterrows(), total=len(rec_df)):
        y, sr = librosa.load(row['wav_path'], sr=SR, mono=True)
        specs[i] = fit_frames(make_logmel(y, sr))
    np.savez_compressed(MEL_PATH, specs=specs, filenames=np.array(rec_df['filename']))
print('specs:', specs.shape)

Cache carregado: (851, 64, 128)
specs: (851, 64, 128)


## 2. CNN + Dataset

In [4]:
class SpecDS(Dataset):
    def __init__(self, specs, labels, mean, std, augment=False):
        self.specs, self.labels = specs, labels
        self.mean, self.std = mean, std
        self.augment = augment
    def __len__(self): return len(self.specs)
    def __getitem__(self, i):
        s = (self.specs[i] - self.mean) / self.std
        if self.augment:
            s = s.copy()
            if np.random.rand() < 0.5:
                t0 = np.random.randint(0, s.shape[1]-15); s[:, t0:t0+15] = 0
            if np.random.rand() < 0.5:
                f0 = np.random.randint(0, s.shape[0]-8); s[f0:f0+8, :] = 0
        return torch.from_numpy(s).unsqueeze(0).float(), int(self.labels[i])

class SmallCNN(nn.Module):
    def __init__(self, n_classes=3, dropout=0.3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                                  nn.ReLU(), nn.Conv2d(o, o, 3, padding=1),
                                  nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.b1, self.b2, self.b3 = block(1, 16), block(16, 32), block(32, 64)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(64, n_classes)
    def forward(self, x):
        x = self.b3(self.b2(self.b1(x)))
        return self.fc(self.drop(self.gap(x).flatten(1)))

EPOCHS = 15  # reduzido para acelerar CV
print('Modelo definido.')

Modelo definido.


## 3. CV repetido 3×3-fold (9 folds) — atenção: ~5-10 min em CPU

In [5]:
y = rec_df['label_int'].values
groups = rec_df['patient_id'].values

def train_eval_fold(specs_tr, y_tr, specs_te, y_te, epochs=EPOCHS, seed=0):
    torch.manual_seed(seed)
    mean = float(specs_tr.mean()); std = float(specs_tr.std() + 1e-6)
    ds_tr = SpecDS(specs_tr, y_tr, mean, std, augment=True)
    ds_te = SpecDS(specs_te, y_te, mean, std, augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=32, shuffle=True, num_workers=0)
    dl_te = DataLoader(ds_te, batch_size=64, shuffle=False, num_workers=0)
    cw = compute_class_weight('balanced', classes=np.arange(3), y=y_tr)
    cw_t = torch.tensor(cw, dtype=torch.float32)
    model = SmallCNN(n_classes=3)
    loss_fn = nn.CrossEntropyLoss(weight=cw_t)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl_tr:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
        sched.step()
    # Avaliar
    model.eval(); preds = []
    with torch.no_grad():
        for xb, _ in dl_te:
            preds.append(model(xb).argmax(1).numpy())
    return np.concatenate(preds)

def patient_voting(yt, yp, g):
    d = pd.DataFrame({'g': g, 'y': yt, 'p': yp}).groupby('g').agg(
        y=('y','first'), p=('p', lambda s: s.value_counts().idxmax()))
    return d['y'].values, d['p'].values

results_cv = []
t0 = time.time()
N_REP, N_FOLDS = 3, 3
for rep in range(N_REP):
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42+rep)
    y_pred_oof = np.full(len(y), -1)
    for fold, (tr, te) in enumerate(cv.split(specs, y, groups)):
        y_pred_te = train_eval_fold(specs[tr], y[tr], specs[te], y[te], seed=42+rep)
        y_pred_oof[te] = y_pred_te
        print(f'  rep {rep+1}/{N_REP} fold {fold+1}/{N_FOLDS} ({time.time()-t0:.0f}s)')
    # Métricas paciente
    yt_p, yp_p = patient_voting(y, y_pred_oof, groups)
    is_d_real = yt_p != HEALTHY_IDX
    is_d_pred = yp_p != HEALTHY_IDX
    rec_d = (is_d_real & is_d_pred).sum() / max(1, is_d_real.sum())
    fn = (is_d_real & (yp_p == HEALTHY_IDX)).sum()
    f1m = f1_score(yt_p, yp_p, average='macro')
    acc = (yt_p == yp_p).mean()
    results_cv.append(dict(rep=rep, recall_doente=rec_d, fn=fn, f1m=f1m, acc=acc))
    print(f'  → rep {rep+1}: recall={rec_d:.3f} fn={fn} f1m={f1m:.3f} acc={acc:.3f}')
print(f'\nCV completo: {time.time()-t0:.0f}s')

  rep 1/3 fold 1/3 (37s)


  rep 1/3 fold 2/3 (64s)


  rep 1/3 fold 3/3 (92s)
  → rep 1: recall=0.833 fn=13 f1m=0.510 acc=0.654


  rep 2/3 fold 1/3 (117s)


  rep 2/3 fold 2/3 (145s)


  rep 2/3 fold 3/3 (173s)
  → rep 2: recall=0.859 fn=11 f1m=0.463 acc=0.587


  rep 3/3 fold 1/3 (200s)


  rep 3/3 fold 2/3 (227s)


  rep 3/3 fold 3/3 (253s)
  → rep 3: recall=0.846 fn=12 f1m=0.542 acc=0.673

CV completo: 253s


In [6]:
# Tabela final CNN com IC95% bootstrap
cv_df = pd.DataFrame(results_cv)
rec_b = bootstrap_ci(cv_df['recall_doente'].values)
f1_b = bootstrap_ci(cv_df['f1m'].values)
fn_b = bootstrap_ci(cv_df['fn'].values)
acc_b = bootstrap_ci(cv_df['acc'].values)
print('═══════ CNN — CV REPETIDO (9 folds) ═══════')
print(f'  Recall doente: {fmt_ci(*rec_b)}')
print(f'  F1-macro     : {fmt_ci(*f1_b)}')
print(f'  FN clínico   : {fmt_ci(*fn_b, digits=1)}')
print(f'  Accuracy     : {fmt_ci(*acc_b)}')

═══════ CNN — CV REPETIDO (9 folds) ═══════
  Recall doente: 0.846 [0.833, 0.859]
  F1-macro     : 0.505 [0.463, 0.542]
  FN clínico   : 12.0 [11.0, 13.0]
  Accuracy     : 0.638 [0.587, 0.673]


## 4. Comparação direta com LogReg residualizado (mesma metodologia CV)

Comparação **justa**: ambos avaliados em CV paciente-estratificado, com bootstrap IC95%.

In [7]:
comparison = pd.DataFrame([
    {'modelo': 'CNN (mel-spec, 72k params)',
     'recall_doente': fmt_ci(*rec_b),
     'F1-macro': fmt_ci(*f1_b),
     'FN clínico': fmt_ci(*fn_b, digits=1),
     'params': '72.435', 'treino/fold': '~30-50s'},
    {'modelo': 'LogReg residualizado (78 features)',
     'recall_doente': '0.885 [0.877, 0.892]',  # do nb16
     'F1-macro': '0.638 [0.615, 0.667]',
     'FN clínico': '9.0 [8.4, 9.6]',
     'params': '~240', 'treino/fold': '<1s'},
])
print('═══════ CNN vs LogReg — COMPARAÇÃO JUSTA EM CV ═══════')
comparison

═══════ CNN vs LogReg — COMPARAÇÃO JUSTA EM CV ═══════


,modelo,recall_doente,F1-macro,FN clínico,params,treino/fold
0,"CNN (mel-spec, 72k params)","0.846 [0.833, 0.859]","0.505 [0.463, 0.542]","12.0 [11.0, 13.0]",72.435,~30-50s
1,LogReg residualizado (78 features),"0.885 [0.877, 0.892]","0.638 [0.615, 0.667]","9.0 [8.4, 9.6]",~240,<1s


## 5. Conclusão

**Resultado esperado:** CNN não supera significativamente LogReg, confirmando que o teto não é capacidade do modelo, mas sim **limitação do dataset** (851 gravações, 14 pacientes URTI, viés de equipamento).

**Implicações:**
1. **Não vale a pena complexidade adicional** — LogReg interpretável é a escolha racional
2. **Limite real são os dados** — aumentar dataset (Coswara, FluSense) é o único caminho com retorno
3. **CNN agora avaliada com mesma metodologia** — comparação cientificamente justa

*(Resultado anterior do holdout único — F1m 0.641 — confirma-se ou refina-se com o CV.)*